# Virtual threads (Project Loom, Java 21) vs. platform threads

A platform `Thread` is a thin wrapper around one real OS thread -- expensive enough that thread pools exist
specifically to avoid creating too many. A virtual thread is a JVM-managed lightweight thread; you can spin
up hundreds of thousands of them, and a *blocking* call (like `Thread.sleep`) on a virtual thread doesn't
block an OS thread underneath it -- the JVM parks it and reuses the carrier thread for other virtual threads.

In [1]:
Thread platformThread = Thread.ofPlatform().unstarted(() -> {});
Thread virtualThread = Thread.ofVirtual().unstarted(() -> {});
System.out.println("platform thread isVirtual(): " + platformThread.isVirtual());
System.out.println("virtual thread isVirtual(): " + virtualThread.isVirtual());


platform thread isVirtual(): false


virtual thread isVirtual(): true


The practical payoff: 10,000 concurrent "blocking" sleeps finish together, not sequentially -- something that would exhaust a platform thread pool almost immediately:

In [2]:
import java.util.concurrent.*;

int numTasks = 10_000;
var latch = new CountDownLatch(numTasks);
long start = System.nanoTime();

try (ExecutorService executor = Executors.newVirtualThreadPerTaskExecutor()) {
    for (int i = 0; i < numTasks; i++) {
        executor.submit(() -> {
            try { Thread.sleep(20); } catch (InterruptedException ignored) {}
            latch.countDown();
        });
    }
    latch.await();
}

long elapsedMs = (System.nanoTime() - start) / 1_000_000;
System.out.println(numTasks + " virtual threads, each \"blocking\" for 20ms, all finished in ~" + elapsedMs
    + "ms -- not the ~200 seconds a small platform-thread pool running them sequentially would need");


10000 virtual threads, each "blocking" for 20ms, all finished in ~177ms -- not the ~200 seconds a small platform-thread pool running them sequentially would need
